18a. Chạy thử bộ tổng hợp GĐ H trên đầu ra val của 17a, CPU (không đọc test)

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from pathlib import Path

ALLOW_GPU = False                  # toàn bộ quy trình chạy được trên CPU (suy luận và huấn luyện baseline cây); không dùng GPU
SPLIT = "aggregate"                # "val" = chạy thử trước khi đóng băng; "test" = chạy test một lần
REF = ""                    # "" = nhánh main (chạy thử); "freeze-v1" = bản đóng băng
WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "split": SPLIT, "ref": REF or "main"}
T_START = time.time()


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = round(INFO["total_s"] / 3600, 3) if INFO["device"] == "GPU" else 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    """Chạy lệnh; có `log` thì ghi toàn bộ đầu ra vào file đó (in 6000 ký tự cuối), không thì in 9000 ký tự cuối."""
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, **kw)
        print(open(log, encoding="utf-8").read()[-6000:])
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, **kw)
        print(r.stdout[-9000:], end="")
        print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    """Đường dẫn đầu tiên khớp glob đệ quy dưới /kaggle/input (báo lỗi rõ nếu không có)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", *(["--branch", REF] if REF else []), URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    if REF:                                         # commit đã clone phải đúng commit của tag trên GitHub
        lines = subprocess.run(["git", "ls-remote", URL, f"refs/tags/{REF}", f"refs/tags/{REF}^{{}}"], capture_output=True, text=True).stdout.split("\n")
        tag_sha = [l.split()[0] for l in lines if l.strip()][-1]
        assert tag_sha == INFO["commit"], f"Commit đã clone {INFO['commit']} khác commit của tag {REF}: {tag_sha}"
        INFO["tag"], INFO["tag_commit"] = REF, tag_sha
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| tag:", INFO.get("tag"), "| máy:", INFO["device"], INFO["gpu"], "| split:", SPLIT)
assert ALLOW_GPU or INFO["device"] == "CPU", "Không dùng GPU cho notebook này: tắt GPU trong cài đặt notebook"

## pytest toàn bộ (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    # test tái lập hai lần chạy độc lập không ổn định trên GPU (scatter không tất định); nó đã pass trên CPU ở notebook 16t và 16a
    r = run([sys.executable, "-m", "pytest", "-q", "--deselect", "tests/test_hybrid.py::test_run_hybrid_end_to_end_both_modes_val_only_reproducible"] if INFO["device"] == "GPU" else [sys.executable, "-m", "pytest", "-q"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## Đường dẫn đầu vào
- Đầu ra của notebook 17 (test) hoặc 17a (chạy thử trên val): kết quả, chỉ số theo tầng, dự đoán từng node.
- Đầu ra của 17a: dòng `val` của mọi mô hình cuối (cùng mã, cùng mô hình với dòng test).
- Đồ thị đã dựng (04, 11): chỉ để đọc `split.json` (tên lớp). Đường học, attention, ngưỡng: từ bản repo đã kéo về (`reports/kaggle/`).

In [ ]:
EVAL_SPLIT = "val"            # "test" (thật) hoặc "val" (chạy thử bộ tổng hợp trên đầu ra của 17a)
EVAL_SLUG = "gnn-iot-ids-17a-dryrun-val"             # notebook cung cấp dòng đánh giá: gnn-iot-ids-17-final-test hoặc gnn-iot-ids-17a-dryrun-val
DS_T, DS_B = "nf_ton_iot_v3", "nf_bot_iot_v3"
EVAL_DIR = os.path.dirname(find(f"{EVAL_SLUG}/results_{EVAL_SPLIT}_{DS_T}.csv"))
VAL_DIR = os.path.dirname(find(f"gnn-iot-ids-17a-dryrun-val/results_val_{DS_T}.csv"))
CFG = dict(eval_split=EVAL_SPLIT, eval_dir=EVAL_DIR, val_dir=VAL_DIR, repo=REPO, out=WORK,
           data={DS_T: os.path.dirname(find(f"{DS_T}/split.json")), DS_B: os.path.dirname(find(f"{DS_B}/split.json"))})
json.dump(CFG, open(f"{WORK}/aggregate_config.json", "w"), ensure_ascii=False, indent=1)
INFO["config"] = CFG
save_info()
print(json.dumps(CFG, indent=1, ensure_ascii=False))
print(sorted(os.listdir(EVAL_DIR))[:12], "...", len(os.listdir(EVAL_DIR)), "file")

## Tổng hợp: bảng, McNemar, chỉ tiêu, phân tích lỗi, hình

In [ ]:
with step("aggregate_h"):
    run([sys.executable, "-I", "scripts/aggregate_h.py", f"{WORK}/aggregate_config.json"], log=f"{WORK}/aggregate_h_log.txt", cwd=REPO)

## Kiểm tra đầu ra

In [ ]:
import pandas as pd
res = pd.read_csv(f"{WORK}/results.csv")
INFO["check"] = {
    "results_rows": len(res), "results_columns": res.shape[1], "splits": sorted(res["split"].unique().tolist()),
    "commits": res.groupby("split")["commit"].unique().apply(list).to_dict(),
    "figures": sorted(f for f in os.listdir(WORK) if f.startswith("gd6_") and f.endswith(".png")),
    "tables": sorted(f for f in os.listdir(WORK) if f.endswith(".csv")),
}
assert res.shape[1] == 20
save_info()
print(json.dumps(INFO["check"], indent=1, ensure_ascii=False))
print(pd.read_csv(f"{WORK}/targets.csv")[["target", "dataset", "model", "mode", "value", "threshold", "result"]].to_string())
print(open(f"{WORK}/run_info.json", encoding="utf-8").read()[-1500:])